# Data Cleaning
## Movies and Box Office Dataset

In [ ]:
# %%
import pandas as pd

# Load IMDb datasets 
imdb_basics = pd.read_csv("../data/raw/title.basics.tsv", sep="\t", nrows=5000)
imdb_ratings = pd.read_csv("../data/raw/title.ratings.tsv", sep="\t", nrows=5000)

# Load box office dataset
boxoffice = pd.read_csv("../data/raw/enhanced_box_office_data(2000-2024)u.csv")

# Load Rotten Tomatoes dataset
rotten = pd.read_csv("rotten_tomatoes_critic_reviews.csv")

print("Datasets loaded successfully")

print(imdb_basics.head())
print(imdb_ratings.head())
print(boxoffice.head())
print(rotten.head())

# %%
# Keep only movies
imdb_movies = imdb_basics[imdb_basics["titleType"] == "movie"]

# Remove missing years
imdb_movies = imdb_movies.dropna(subset=["startYear"])

# Merge ratings with movie data
movies = imdb_movies.merge(imdb_ratings, on="tconst", how="left")

print(movies.head())

# %%
import re

def clean_title(title):
    title = str(title).lower()
    title = re.sub(r"[^\w\s]", "", title)
    return title.strip()

movies["clean_title"] = movies["primaryTitle"].apply(clean_title)

# Save cleaned dataset
movies.to_csv("movies_clean.csv", index=False)

print("Clean dataset saved as movies_clean.csv")

# %%
# Clean box office dataset

# remove rows without worldwide revenue
boxoffice = boxoffice.dropna(subset=["$Worldwide"])

# remove duplicates
boxoffice = boxoffice.drop_duplicates()

# create clean movie title column
boxoffice["clean_title"] = boxoffice["Release Group"].astype(str).str.lower()

# save cleaned dataset
boxoffice.to_csv("boxoffice_clean.csv", index=False)

print("boxoffice_clean.csv saved")

# %%

## Rotten Tomatoes Dataset

In [3]:
# Load Rotten Tomatoes File
import pandas as pd
import re

# Load CSVs
movies_df = pd.read_csv("data/raw/rotten_tomatoes_movies.csv")  # has movie_title, critics_consensus, link
reviews_df = pd.read_csv("data/raw/rotten_tomatoes_critic_reviews.csv")  # has link, tomatometer, review_date

# Inspect columns to be sure
print(movies_df.columns)
print(reviews_df.columns)

# Select only needed columns (update with actual names from your CSV)
movies_df = movies_df[["rotten_tomatoes_link", "movie_title", "critics_consensus", "tomatometer_rating"]]
reviews_df = reviews_df[["rotten_tomatoes_link", "review_date"]]

# Normalize titles for later join
def normalize_title(title):
    title = title.lower().strip()
    title = re.sub(r"[^\w\s]", "", title)
    title = re.sub(r"\s+", " ", title)
    return title

movies_df["norm_title"] = movies_df["movie_title"].apply(normalize_title)

# Merge on Rotten Tomatoes link (reliable)
df = pd.merge(
    movies_df,
    reviews_df,
    on="rotten_tomatoes_link",
    how="inner"
)

df = df.drop_duplicates(subset=["rotten_tomatoes_link"], keep="first")

# Keep only rows where critics_consensus exists
df = df[df["critics_consensus"].notna()]

# Rename columns to match your table
df = df.rename(columns={
    "tomatometer_rating": "tomatometer_rating"
})

# Extract year from review_date
df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")
df["year"] = df["review_date"].dt.year

# Final columns for rt_reviews table
df_final = df[[
    "movie_title",
    "norm_title",
    "year",
    "critics_consensus",
    "review_date",
    "tomatometer_rating"
]]

# Save cleaned CSV
df_final.to_csv("rt_reviews_clean.csv", index=False)

print("Cleaned Rotten Tomatoes dataset ready.")
print(df_final.head())

Index(['rotten_tomatoes_link', 'movie_title', 'movie_info',
       'critics_consensus', 'content_rating', 'genres', 'directors', 'authors',
       'actors', 'original_release_date', 'streaming_release_date', 'runtime',
       'production_company', 'tomatometer_status', 'tomatometer_rating',
       'tomatometer_count', 'audience_status', 'audience_rating',
       'audience_count', 'tomatometer_top_critics_count',
       'tomatometer_fresh_critics_count', 'tomatometer_rotten_critics_count'],
      dtype='object')
Index(['rotten_tomatoes_link', 'critic_name', 'top_critic', 'publisher_name',
       'review_type', 'review_score', 'review_date', 'review_content'],
      dtype='object')
Cleaned Rotten Tomatoes dataset ready.
                                           movie_title  \
0    Percy Jackson & the Olympians: The Lightning T...   
149                                        Please Give   
291                                                 10   
315                    12 Angry Men (Twe

## Movies

In [12]:
movies = pd.read_csv("data/clean/movies_clean.csv")

In [10]:
movies.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,clean_title
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,\N,45,Romance,5.3,235.0,miss jerry
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897,\N,100,"Documentary,News,Sport",5.3,595.0,the corbettfitzsimmons fight
2,tt0000502,movie,Bohemios,Bohemios,0,1905,\N,100,\N,3.5,26.0,bohemios
3,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,\N,70,"Action,Adventure,Biography",6.0,1064.0,the story of the kelly gang
4,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907,\N,90,Drama,5.0,39.0,the prodigal son


In [11]:
print(list(movies.columns))

['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'clean_title']


## Box Office

In [4]:
box_office = pd.read_csv("data/clean/boxoffice_clean.csv")

In [5]:
print(list(box_office.columns))

['Rank', 'Release Group', '$Worldwide', '$Domestic', 'Domestic %', '$Foreign', 'Foreign %', 'Year', 'Genres', 'Rating', 'Vote_Count', 'Original_Language', 'Production_Countries', 'clean_title']


## Rotten Tomatoes Reviews

In [6]:
rt_reviews = pd.read_csv("data/clean/rt_reviews_clean.csv")

In [7]:
print(list(rt_reviews.columns))

['movie_title', 'norm_title', 'year', 'critics_consensus', 'review_date', 'tomatometer_rating']


The column names of all dataframe are printed to select the one that are relevant to our database design and research questions. These chosen columns will become the attributes in our SQL tables.